# B0 full-gene baseline — Kaggle experiment-dataset-v2

Notebook này chạy baseline **B0_all_genes**: không feature selection, không feature extraction; classifier nhận toàn bộ gene của từng training fold. Mục tiêu là tạo mốc đối chiếu công bằng cho kết quả chọn gene của permutation v3.

- Đọc trực tiếp Kaggle input `/kaggle/input/experiment-dataset-v2/<dataset>/fold_<r>/{train,val,test}.h5` (cũng hỗ trợ biến thể có thư mục `h5/`).
- Chạy đủ bảy dataset hiện có: bốn GEO LUAD-vs-LUSC, GSE68465 stage tumor-only, TCGA LUAD-vs-LUSC và TCGA Normal-vs-Tumor sanity.
- Không tối ưu hyperparameter từ validation; LR và Linear SVM dùng `C=1.0`. Hỗ trợ cả binary LUAD/LUSC và multiclass GSE68465 stage. `balanced` là phân tích chính, `none` là sensitivity analysis.
- Kết quả được ghi riêng vào `results/b0_full_gene_v2/<dataset>/`; không ghi đè các kết quả classifier hay permutation.


In [ ]:
import json, os, platform, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.svm import SVC

RUN_DATASETS = [
    'gse10245_luad_lusc', 'gse30219_luad_lusc', 'gse37745_luad_lusc',
    'gse42127_luad_lusc', 'gse68465_stage_tumor_only',
    'tcga_luad_lusc_primary_tumor', 'tcga_normal_vs_tumor_sanity',
]
CLASS_WEIGHTS = ['balanced', 'none']
MAIN_CW = 'balanced'
SEED = 42
LR_C = SVM_C = 1.0
GIT_COMMIT = ''  # Điền commit hash nếu repository đã được Git hóa.

def split_file(dataset_dir, fold, split):
    candidates = [dataset_dir / f'fold_{fold}' / f'{split}.h5',
                  dataset_dir / 'folds' / f'fold_{fold}' / f'{split}.h5']
    return next((path for path in candidates if path.is_file()), None)

def find_data_root():
    candidates = [
        Path('/kaggle/input/datasets/lfreedom2750/experiment-dataset-v2'),
        Path('/kaggle/input/experiment-dataset-v2'),
        Path('/kaggle/input/experiment-dataset-v2/h5'),
        Path('/kaggle/input/experiment-dataset-v2/datasets'),
        Path('D:/KLTN/experiment-data-v2/h5'),
        Path('experiment-data-v2/h5'),
        Path('/content/experiment-data-v2/h5'),
    ]
    # Kaggle có thể đổi slug mount (ví dụ thêm hậu tố -1), nên quét marker thay vì tin vào tên hiển thị.
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        marker = 'gse10245_luad_lusc/fold_1/train.h5'
        for found in kaggle_input.rglob('train.h5'):
            normalized = found.as_posix()
            if normalized.endswith(marker):
                discovered_root = found.parents[2]  # <root>/gse10245_luad_lusc/fold_1/train.h5
                if discovered_root not in candidates:
                    candidates.insert(0, discovered_root)
                    print(f'Đã tự tìm thấy Kaggle data root: {discovered_root}')
    reports = []
    for root in candidates:
        present = [name for name in RUN_DATASETS if (root / name).is_dir()]
        ready = [name for name in present if split_file(root / name, 1, 'train')]
        reports.append((root, present, ready))
        if len(ready) == len(RUN_DATASETS):
            return root
    # Kaggle có thể lưu trực tiếp một bản subset. Dùng root có nhiều dataset sẵn sàng nhất và báo rõ phần thiếu.
    root, present, ready = max(reports, key=lambda item: (len(item[2]), len(item[1])))
    if ready:
        missing = sorted(set(RUN_DATASETS) - set(ready))
        print(f'CANH BAO: chỉ tìm thấy {len(ready)}/{len(RUN_DATASETS)} dataset có fold. Thiếu/chưa đúng cấu trúc: {missing}')
        return root
    details = '\n'.join(f'- {root}: folders={present}, fold_1 train={ready}' for root, present, ready in reports)
    raise FileNotFoundError('Không tìm thấy H5 train của bất kỳ dataset nào. Đã kiểm tra:\n' + details)

DATA_ROOT = find_data_root()
OUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('results')
RESULT_ROOT = OUT_ROOT / 'b0_full_gene_v2'
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if not GIT_COMMIT:
    try:
        GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        GIT_COMMIT = 'unknown'
print('Data:', DATA_ROOT)
print('Output:', RESULT_ROOT)


## Hàm baseline

`B0_lr` và `B0_linear_svm` dùng toàn bộ gene. `B0_pam_all_genes` là nearest shrunken centroids với ngưỡng bằng 0, tức là cũng dùng toàn bộ gene. Mỗi model được fit lại ở từng training fold; dữ liệu giữ nguyên log2 gốc, không chuẩn hoá thêm, giống v3.


In [ ]:
CW = {'balanced': 'balanced', 'none': None}

class PAMAllGenes:
    def __init__(self, class_weight):
        self.class_weight = class_weight

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        encoded = np.searchsorted(self.classes_, y)
        n, n_classes = len(y), len(self.classes_)
        counts = np.bincount(encoded)
        self.centroids_ = np.stack([X[encoded == i].mean(axis=0) for i in range(n_classes)])
        residual = X - self.centroids_[encoded]
        scale = np.sqrt((residual ** 2).sum(axis=0) / max(n - n_classes, 1))
        self.scale2_ = (scale + np.median(scale)) ** 2
        self.log_prior_ = (np.full(n_classes, -np.log(n_classes)) if self.class_weight == 'balanced'
                           else np.log(counts / n))
        return self

    def predict(self, X):
        scores = -(((X[:, None, :] - self.centroids_[None, :, :]) ** 2) / self.scale2_).sum(axis=2) + 2 * self.log_prior_
        return self.classes_[scores.argmax(axis=1)]

def build_models(class_weight):
    return {
        'B0_lr': LogisticRegression(C=LR_C, max_iter=5000, class_weight=CW[class_weight]),
        'B0_linear_svm': SVC(kernel='linear', C=SVM_C, class_weight=CW[class_weight]),
        'B0_pam_all_genes': PAMAllGenes(class_weight),
    }

def load_dataset(name):
    dataset_dir = DATA_ROOT / name
    meta = pd.read_csv(dataset_dir / 'metadata.csv').set_index('sample_id')
    if 'label' not in meta.columns:
        raise ValueError(f'{name}: metadata.csv thiếu cột label')
    return dataset_dir, meta

def read_split(dataset_dir, meta, fold, split):
    path = split_file(dataset_dir, fold, split)
    if path is None:
        raise FileNotFoundError(f'{dataset_dir.name}: thiếu fold {fold} / {split}.h5; hỗ trợ fold_<r>/ và folds/fold_<r>/')
    frame = pd.read_hdf(path, key='data')
    if 'label' not in frame.columns:
        raise ValueError(f'{dataset_dir.name}, fold {fold}, {split}: H5 thiếu cột label')
    y_h5 = frame.pop('label').to_numpy(dtype=int)  # Bắt buộc tách nhãn, tránh label leakage.
    missing = frame.index.difference(meta.index)
    if len(missing):
        raise ValueError(f'{dataset_dir.name}, fold {fold}, {split}: {len(missing)} sample không có metadata')
    y_meta = meta.loc[frame.index, 'label'].to_numpy(dtype=int)
    if not np.array_equal(y_h5, y_meta):
        raise ValueError(f'{dataset_dir.name}, fold {fold}, {split}: label trong H5 và metadata không khớp')
    return frame.to_numpy(dtype=np.float64, copy=False), y_h5

def evaluate_dataset(name):
    dataset_dir, meta = load_dataset(name)
    rows, conf_rows = [], []
    for fold in range(1, 6):
        Xtr, ytr = read_split(dataset_dir, meta, fold, 'train')
        Xva, yva = read_split(dataset_dir, meta, fold, 'val')
        Xte, yte = read_split(dataset_dir, meta, fold, 'test')
        if len(np.unique(ytr)) < 2:
            raise ValueError(f'{name}, fold {fold}: training split phải có ít nhất hai lớp')
        if not (Xtr.shape[1] == Xva.shape[1] == Xte.shape[1]):
            raise ValueError(f'{name}, fold {fold}: số feature giữa train/val/test không khớp')
        for weight in CLASS_WEIGHTS:
            for model_name, model in build_models(weight).items():
                model.fit(Xtr, ytr)
                for split, Xsplit, ysplit in [('val', Xva, yva), ('test', Xte, yte)]:
                    pred = model.predict(Xsplit)
                    rows.append({'dataset': name, 'fold': fold, 'split': split, 'class_weight': weight,
                                 'method': 'B0_all_genes', 'model': model_name, 'n_features': Xtr.shape[1],
                                 'bacc': balanced_accuracy_score(ysplit, pred), 'seed': SEED})
                    if split == 'test':
                        labels = sorted(meta['label'].unique())
                        cm = confusion_matrix(ysplit, pred, labels=labels)
                        for true_i, true_label in enumerate(labels):
                            for pred_i, pred_label in enumerate(labels):
                                conf_rows.append({'dataset': name, 'fold': fold, 'class_weight': weight,
                                                  'model': model_name, 'true': true_label, 'pred': pred_label,
                                                  'count': int(cm[true_i, pred_i])})
    out = RESULT_ROOT / name
    out.mkdir(parents=True, exist_ok=True)
    scores = pd.DataFrame(rows)
    scores.to_csv(out / 'b0_scores_per_fold.csv', index=False)
    pd.DataFrame(conf_rows).to_csv(out / 'b0_confusion.csv', index=False)
    summary = (scores.groupby(['dataset', 'split', 'class_weight', 'method', 'model', 'n_features'])['bacc']
               .agg(['mean', 'std', 'count']).reset_index()
               .rename(columns={'mean': 'bacc_mean', 'std': 'bacc_sd_folds', 'count': 'n_folds'}))
    summary.to_csv(out / 'b0_summary.csv', index=False)
    config = {'dataset': name, 'method': 'B0_all_genes', 'n_features': int(Xtr.shape[1]), 'seed': SEED,
              'n_splits': 5, 'n_classes': int(meta['label'].nunique()),
              'fold_source': 'fold_<r>/{train,val,test}.h5', 'fit_split': 'train',
              'eval_split': 'test (primary); val (reported only)', 'scaling': 'none (raw log2)',
              'models': list(build_models('balanced')), 'class_weights': CLASS_WEIGHTS,
              'lr_c': LR_C, 'svm_c': SVM_C, 'data_dir': str(dataset_dir),
              'commit_hash': GIT_COMMIT, 'versions': {'python': platform.python_version(),
              'sklearn': sklearn.__version__, 'numpy': np.__version__, 'pandas': pd.__version__}}
    (out / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    return summary


## Chạy và kiểm tra kết quả

Sau khi chạy, chỉ dùng hàng `split == test` và `class_weight == balanced` làm kết quả B0 chính. Hàng validation chỉ là chẩn đoán, không dùng để chọn model hoặc thay đổi tham số.


In [ ]:
all_summaries = []
for dataset in RUN_DATASETS:
    print(f'\n=== {dataset} ===')
    all_summaries.append(evaluate_dataset(dataset))

aggregate = pd.concat(all_summaries, ignore_index=True)
aggregate.to_csv(RESULT_ROOT / 'all_b0_summary.csv', index=False)
display(aggregate.query("split == 'test' and class_weight == @MAIN_CW").sort_values(['dataset', 'model']))
print(f'Đã ghi B0 v3 vào: {RESULT_ROOT}')


## Nén kết quả để tải từ Kaggle


In [ ]:
import shutil

if not (RESULT_ROOT / 'all_b0_summary.csv').is_file():
    raise FileNotFoundError('Chưa có all_b0_summary.csv. Hãy chạy cell thực nghiệm trước khi nén.')
zip_path = shutil.make_archive(str(OUT_ROOT / 'b0_full_gene_v2'), 'zip', root_dir=OUT_ROOT, base_dir=RESULT_ROOT.name)
print(f'Đã tạo ZIP: {zip_path}')
